# Final Test Evaluation — Social Science Concept Integration (APA)

**Purpose:** This notebook performs the definitive evaluation of ML models for concept harmonisation in social science using the **full APA dataset** (train + test merged).  
APA was not included in the cross-validation calibration process, so all available pairs are evaluated here.  
Each model is evaluated under **three techniques** (Clustering, Pairwise, Seeded Clustering) and subjected to **five psychometric audits**:

| # | Audit | What it measures |
|---|-------|-----------------|
| 1 | **Reliability** | Standard Error of Model (SEM) under stochastic embedding noise |
| 2 | **Discriminant Validity** | False-positive rate on lexically similar negative pairs |
| 3 | **DIF (Rare-Word Bias)** | Recall gap between common and rare terminology |
| 4 | **Semantic Decay** | Recall degradation as keyword overlap vanishes |
| 5 | **Structural Validity** | Correlation with expert APA graph distances (Spearman, Pearson, Point-Biserial) |

> **Reproducibility:** All random operations use `SEED = 42`. Models are loaded from local HuggingFace cache. Best hyperparameters are fixed from prior cross-validation on ELSST.

## 0. Environment Setup & Imports

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*n_jobs value 1 overridden.*")
import sys
import os
from pathlib import Path

# Add parent directory to Python path and change working directory
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

# Change working directory to parent so relative paths in config work correctly  
os.chdir(parent_dir)
print(f"Changed working directory to: {os.getcwd()}")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from contextlib import redirect_stdout
from io import StringIO
from IPython.display import display, Markdown

# Local project modules
from scripts import model_utils_shared as shared
from scripts import model_utils_clustering as muc
from scripts import model_utils_pairwise as mup
from scripts import model_utils_seed as mus
from scripts import config

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"Random seed     : {config.SEED}")

Changed working directory to: /home/rass/Desktop/SocialScience-ConceptIntegration


/home/rass/Desktop/SocialScience-ConceptIntegration/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version : 2.9.1+cpu
CUDA available  : False
Random seed     : 42


## 1. Configuration — Best Hyperparameters (from Cross-Validation)

These parameters were determined during the training phase via grid search / cross-validation.  
They are **fixed** for this final held-out evaluation and must **not** be modified.

In [2]:
# ── Models to evaluate ────────────────────────────────────────────────────────
MODELS_TO_EVAL = [
    "all-mpnet-base-v2",
    "dwulff/mpnet-personality",
    "allenai/scibert_scivocab_uncased",
    "bert-base-uncased",
]

# ── Best HDBSCAN clustering parameters (from training grid search) ───────────
BEST_CLUSTERING_PARAMS = config.BEST_CLUSTERING_PARAMS
# ── Best pairwise cosine similarity thresholds ───────────────────────────────
BEST_PAIRWISE_THRESHOLDS = config.BEST_PAIRWISE_THRESHOLDS

# ── Best seeded clustering parameters ────────────────────────────────────────
BEST_SEEDED_PARAMS = config.BEST_SEEDED_PARAMS

# ── APA Data paths (full dataset — train + test merged) ──────────────────────
# APA was not part of calibration/cross-validation, so the full dataset is used
TRAIN_POS_PATH = "datasets/processed_datasets/apa/train_positive_pairs.csv"
TRAIN_NEG_PATH = "datasets/processed_datasets/apa/train_negative_pairs.csv"
TEST_POS_PATH  = "datasets/processed_datasets/apa/test_positive_pairs.csv"
TEST_NEG_PATH  = "datasets/processed_datasets/apa/test_negative_pairs.csv"

# ── Audit parameters ────────────────────────────────────────────────────────
NOISE_LEVELS = config.NOISE_LEVELS
N_RELIABILITY_RUNS = config.N_RELIABILITY_RUNS

print("✓ Configuration loaded.")


✓ Configuration loaded.


## 2. Load Full APA Data (Train + Test Merged) & Prepare Indices

We load **both the training and test splits** of the APA dataset and merge them, since APA was not part of the cross-validation calibration process. All downstream indices and audit masks are built on this merged full dataset.

In [3]:
shared.setup_reproducibility(config.SEED)

# Load and merge train + test data
# APA was not used in calibration/cross-validation, so we evaluate on the full dataset
train_pos = pd.read_csv(TRAIN_POS_PATH, on_bad_lines="skip")
train_neg = pd.read_csv(TRAIN_NEG_PATH, on_bad_lines="skip")
test_pos  = pd.read_csv(TEST_POS_PATH,  on_bad_lines="skip")
test_neg  = pd.read_csv(TEST_NEG_PATH,  on_bad_lines="skip")

for df in (train_pos, test_pos):
    if "label" not in df.columns:
        df["label"] = 1
for df in (train_neg, test_neg):
    if "label" not in df.columns:
        df["label"] = 0

pos_df = pd.concat([train_pos, test_pos], ignore_index=True)
neg_df = pd.concat([train_neg, test_neg], ignore_index=True)

for col in ("term1", "term2"):
    pos_df[col] = pos_df[col].astype(str)
    neg_df[col] = neg_df[col].astype(str)

full_df = pd.concat([pos_df, neg_df], ignore_index=True)

display(Markdown(f"""
| Split | Count |
|-------|------:|
| Positive pairs (train + test) | {len(pos_df):,} |
| Negative pairs (train + test) | {len(neg_df):,} |
| **Total**                     | **{len(full_df):,}** |
"""))

# Build index arrays
terms1 = full_df["term1"].astype(str).tolist()
terms2 = full_df["term2"].astype(str).tolist()
labels = full_df["label"].astype(int).values
shortest_path = full_df.get("shortest_path", pd.Series([-1] * len(full_df))).values

unique_terms = pd.unique(full_df[["term1", "term2"]].values.ravel("K")).tolist()
term_to_idx = {t: i for i, t in enumerate(unique_terms)}

idx1 = np.array([term_to_idx[t] for t in terms1], dtype=np.int32)
idx2 = np.array([term_to_idx[t] for t in terms2], dtype=np.int32)
idx1_t = torch.tensor(idx1, dtype=torch.long)
idx2_t = torch.tensor(idx2, dtype=torch.long)

pos_len = len(pos_df)

print(f"Unique terms: {len(unique_terms):,}")
print(f"Pairs with expert shortest_path ≥ 0: {(shortest_path >= 0).sum():,}")



| Split | Count |
|-------|------:|
| Positive pairs (train + test) | 5,639 |
| Negative pairs (train + test) | 15,573,509 |
| **Total**                     | **15,579,148** |


Unique terms: 11,014
Pairs with expert shortest_path ≥ 0: 15,387,276


## 3. Precompute Audit Masks

Build boolean masks for the lexical, frequency, and difficulty subsets used by Audits 2–5.  
These are computed once and reused across all models and techniques.

In [4]:
# Letter n-gram cache (3-grams) for lexical similarity
token_cache = {t: shared.letter_ngrams(t, n=3) for t in unique_terms}

# ── Audit 2: Lexical trap masks (hard & easy negatives) ──────────────────────
# Compute Jaccard similarity for all negative pairs
jacc_neg = np.array([
    shared.jaccard_sim(token_cache.get(t1, set()), token_cache.get(t2, set()))
    for t1, t2 in zip(neg_df["term1"], neg_df["term2"])
])

# Hard negatives: lexical traps (high surface similarity, Jaccard > 0.5)
hard_neg_mask_local = jacc_neg > 0.5
hard_neg_mask = np.zeros(len(full_df), dtype=bool)
hard_neg_mask[pos_len:] = hard_neg_mask_local

# Easy negatives: clearly distinct pairs (zero lexical overlap, Jaccard = 0.0)
easy_neg_mask_local = jacc_neg == 0.0
easy_neg_mask = np.zeros(len(full_df), dtype=bool)
easy_neg_mask[pos_len:] = easy_neg_mask_local

# Backward compatibility: lex_mask = hard_neg_mask
lex_mask = hard_neg_mask

# ── Audit 5: Easy / Hard positive masks ──────────────────────────────────────
jacc_pos = np.array([
    shared.jaccard_sim(token_cache.get(t1, set()), token_cache.get(t2, set()))
    for t1, t2 in zip(pos_df["term1"], pos_df["term2"])
])
easy_pos_mask = np.zeros(len(full_df), dtype=bool)
hard_pos_mask = np.zeros(len(full_df), dtype=bool)
easy_pos_mask[:pos_len] = jacc_pos > 0.5
hard_pos_mask[:pos_len] = jacc_pos == 0.0

# Backward compatibility
easy_mask = easy_pos_mask
hard_mask = hard_pos_mask

# ── Audit 4: Rare / Common word frequency masks ─────────────────────────────
try:
    from wordfreq import zipf_frequency

    term_freq = {}
    for t in unique_terms:
        toks = shared.simple_tokens(t)
        term_freq[t] = float(np.mean([zipf_frequency(tok, "en") for tok in toks])) if toks else 0.0

    pos_pair_freq = np.array([
        (term_freq.get(t1, 0.0) + term_freq.get(t2, 0.0)) / 2.0
        for t1, t2 in zip(pos_df["term1"], pos_df["term2"])
    ])
    low_thr = np.quantile(pos_pair_freq, 0.10)
    high_thr = np.quantile(pos_pair_freq, 0.90)

    rare_mask = np.zeros(len(full_df), dtype=bool)
    common_mask = np.zeros(len(full_df), dtype=bool)
    rare_mask[:pos_len] = pos_pair_freq <= low_thr
    common_mask[:pos_len] = pos_pair_freq >= high_thr
    has_wordfreq = True
except ImportError:
    rare_mask = common_mask = np.zeros(len(full_df), dtype=bool)
    has_wordfreq = False
    print("⚠ wordfreq not installed — Audit 4 will be skipped.")

# ── Summary ──────────────────────────────────────────────────────────────────
display(Markdown(f"""
| Audit Mask | N |
|-----------|--:|
| **Negatives** | |
| → Hard negatives (lexical traps, Jaccard > 0.5) | {hard_neg_mask.sum():,} |
| → Easy negatives (zero overlap, Jaccard = 0.0) | {easy_neg_mask.sum():,} |
| **Positives** | |
| → Easy positives (Jaccard > 0.5) | {easy_pos_mask.sum():,} |
| → Hard positives (Jaccard = 0.0) | {hard_pos_mask.sum():,} |
| **Frequency** | |
| → Common pairs (top 10 % freq) | {common_mask.sum():,} |
| → Rare pairs (bottom 10 % freq) | {rare_mask.sum():,} |
"""))



| Audit Mask | N |
|-----------|--:|
| **Negatives** | |
| → Hard negatives (lexical traps, Jaccard > 0.5) | 932 |
| → Easy negatives (zero overlap, Jaccard = 0.0) | 12,886,583 |
| **Positives** | |
| → Easy positives (Jaccard > 0.5) | 1,086 |
| → Hard positives (Jaccard = 0.0) | 1,447 |
| **Frequency** | |
| → Common pairs (top 10 % freq) | 565 |
| → Rare pairs (bottom 10 % freq) | 567 |


## 4. Load Models & Build Embeddings

Each model is loaded once and its embeddings are cached in memory.  
All subsequent technique evaluations and audits reuse these cached embeddings.

In [5]:
embedding_cache = {}

for model_name in MODELS_TO_EVAL:
    model_cfg = config.MODELS[model_name]
    model_type = model_cfg.get("type", config.TYPE_SENTENCE)
    display_name = model_cfg.get("display_name", model_name)

    print(f"\n{'─' * 60}")
    print(f"  {display_name}  ({model_name})")
    print(f"{'─' * 60}")

    model = shared.load_model(model_name=model_name, model_type=model_type)
    if model is None:
        print(f"  ✗ FAILED — skipping")
        continue

    embedding_cache[model_name] = shared.build_embeddings(model, unique_terms, batch_size=config.BATCH_SIZE)
    print(f"  ✓ Embeddings cached — shape {embedding_cache[model_name]['norm_np'].shape}")

    # Free model from GPU memory
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'═' * 60}")
print(f"  Models loaded: {len(embedding_cache)} / {len(MODELS_TO_EVAL)}")
print(f"{'═' * 60}")


────────────────────────────────────────────────────────────
  All-MPNet-Base-v2  (all-mpnet-base-v2)
────────────────────────────────────────────────────────────
Loading Model (all-mpnet-base-v2)...
  Encoding 11014 unique terms...


Batches: 100%|██████████| 689/689 [01:07<00:00, 10.16it/s]


  ✓ Embeddings cached — shape (11014, 768)

────────────────────────────────────────────────────────────
  MPNet-Personality  (dwulff/mpnet-personality)
────────────────────────────────────────────────────────────
Loading Model (dwulff/mpnet-personality)...
  Encoding 11014 unique terms...


Batches: 100%|██████████| 689/689 [01:03<00:00, 10.78it/s]


  ✓ Embeddings cached — shape (11014, 768)

────────────────────────────────────────────────────────────
  SciBERT (SciVocab)  (allenai/scibert_scivocab_uncased)
────────────────────────────────────────────────────────────
Loading Model (allenai/scibert_scivocab_uncased)...
  Encoding 11014 unique terms...


Batches: 100%|██████████| 689/689 [00:54<00:00, 12.70it/s]


  ✓ Embeddings cached — shape (11014, 768)

────────────────────────────────────────────────────────────
  BERT Base  (bert-base-uncased)
────────────────────────────────────────────────────────────
Loading Model (bert-base-uncased)...
  Encoding 11014 unique terms...


Batches: 100%|██████████| 689/689 [01:03<00:00, 10.90it/s]


  ✓ Embeddings cached — shape (11014, 768)

════════════════════════════════════════════════════════════
  Models loaded: 4 / 4
════════════════════════════════════════════════════════════


In [6]:
# Storage for all predictions (used by audits later)
preds_store = {"clustering": {}, "pairwise": {}, "seeded": {}}

---

## 5. Final Evaluation — Three Techniques

We evaluate every model under three harmonisation techniques using their best hyperparameters from cross-validation.  
All predictions are stored in `preds_store` for reuse in the audit sections.


In [7]:
# Storage for all predictions (used by audits later)
preds_store = {"clustering": {}, "pairwise": {}, "seeded": {}}

### 5.1 Clustering (UMAP + HDBSCAN)

Uses `model_utils_clustering.run_hdbscan_clustering()` and `model_utils_clustering.evaluate_clustering()` with the best parameters.

In [8]:

# ── 5.1  CLUSTERING ──────────────────────────────────────────────────────────
clustering_rows = []

for model_name in MODELS_TO_EVAL:
    if model_name not in embedding_cache:
        continue

    emb = embedding_cache[model_name]
    params = BEST_CLUSTERING_PARAMS[model_name]

    # Dimensionality reduction via muc.reduce_embeddings_with_umap()
    n_comp = params.get("n_components")
    if n_comp is None or n_comp >= emb["norm_np"].shape[1]:
        reduced = emb["norm_np"]
    else:
        reduced, _ = muc.reduce_embeddings_with_umap(
            emb["norm_np"], n_components=n_comp, random_state=config.SEED
        )

    # Cluster via muc.run_hdbscan_clustering()
    cluster_labels = muc.run_hdbscan_clustering(
        reduced,
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
    )

    # Evaluate via muc.evaluate_clustering() — returns sklearn metrics + cluster stats
    p, r, f1, pos_acc, neg_acc, n_clusters, n_noise = muc.evaluate_clustering(
        cluster_labels, full_df, np.array(unique_terms)
    )

    # Also extract raw predictions for audit reuse
    l1 = cluster_labels[idx1]
    l2 = cluster_labels[idx2]
    preds = ((l1 != -1) & (l1 == l2)).astype(int)

    clustering_rows.append({
        "Model": config.MODELS[model_name]["display_name"],
        "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "F1": f"{f1:.4f}",
        "Pos Acc": f"{pos_acc:.4f}", "Neg Acc": f"{neg_acc:.4f}",
        "Clusters": n_clusters, "Noise": n_noise,
    })

    preds_store["clustering"][model_name] = {
        "preds": preds, "cluster_labels": cluster_labels,
        "idx1": idx1, "idx2": idx2,
    }

display(Markdown("### Results — Clustering"))
display(pd.DataFrame(clustering_rows).set_index("Model"))


### Results — Clustering

,Precision,Recall,F1,Pos Acc,Neg Acc,Clusters,Noise
Model,,,,,,,
All-MPNet-Base-v2,0.5982,0.3949,0.4758,0.3949,0.9999,2725,3486
MPNet-Personality,0.6092,0.3958,0.4798,0.3958,0.9999,2779,3421
SciBERT (SciVocab),0.3228,0.1887,0.2382,0.1887,0.9999,2338,4364
BERT Base,0.3189,0.1727,0.2241,0.1727,0.9999,2267,4626


### 5.2 Pairwise (Cosine Similarity Thresholding)

Uses `model_utils_pairwise.evaluate_pairwise()` at the best threshold for each model.

In [9]:

# ── 5.2  PAIRWISE ────────────────────────────────────────────────────────────
pairwise_rows = []

for model_name in MODELS_TO_EVAL:
    if model_name not in embedding_cache:
        continue

    term_embeddings = embedding_cache[model_name]["norm_t"]
    threshold = BEST_PAIRWISE_THRESHOLDS[model_name]

    # Cosine similarity via mup.compute_similarities()
    sims = mup.compute_similarities(term_embeddings, term_to_idx, terms1, terms2)

    # Evaluate via mup.evaluate_pairwise() — returns sklearn metrics + MCC
    p, r, f1, pos_acc, neg_acc, mcc = mup.evaluate_pairwise(sims, labels, threshold)

    preds = (sims > threshold).astype(np.int8)

    pairwise_rows.append({
        "Model": config.MODELS[model_name]["display_name"],
        "Threshold": f"{threshold:.2f}",
        "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "F1": f"{f1:.4f}",
        "Pos Acc": f"{pos_acc:.4f}", "Neg Acc": f"{neg_acc:.4f}",
        "MCC": f"{mcc:.4f}",
    })

    preds_store["pairwise"][model_name] = {"preds": preds, "similarities": sims}

display(Markdown("### Results — Pairwise"))
display(pd.DataFrame(pairwise_rows).set_index("Model"))


### Results — Pairwise

,Threshold,Precision,Recall,F1,Pos Acc,Neg Acc,MCC
Model,,,,,,,
All-MPNet-Base-v2,0.69,0.4343,0.5013,0.4654,0.5013,0.9998,0.4664
MPNet-Personality,0.66,0.4889,0.4536,0.4706,0.4536,0.9998,0.4708
SciBERT (SciVocab),0.89,0.1254,0.2289,0.1621,0.2289,0.9994,0.1691
BERT Base,0.90,0.0678,0.1218,0.0871,0.1218,0.9994,0.0904


### 5.3 Seeded Clustering

Uses `model_utils_seed.sample_unique_terms()` and `model_utils_seed.seed_clustering()` with the best number of seeds and threshold.

In [10]:
# ── 5.3  SEEDED CLUSTERING ────────────────────────────────────────────────────
seeded_rows = []

for model_name in MODELS_TO_EVAL:
    if model_name not in embedding_cache:
        continue

    norm_t = embedding_cache[model_name]["norm_t"].cpu().detach()
    term_to_embedding = {t: e for t, e in zip(unique_terms, norm_t)}
    params = BEST_SEEDED_PARAMS[model_name]

    # Sample initial seeds via mus.sample_unique_terms()
    initial_seeds, remaining_terms = mus.sample_unique_terms(
        params["n_initial_seeds"], full_df, np.array(unique_terms), random_state=config.SEED
    )

    # Run seeded clustering via mus.seed_clustering() (suppress verbose output)
    with redirect_stdout(StringIO()):
        seeds_cluster = mus.seed_clustering(
            initial_seeds, remaining_terms, term_to_embedding,
            threshold=params["threshold"],
        )

    # Evaluate via mus.evaluate_seed_clustering() — returns sklearn metrics
    p, r, f1, pos_acc, neg_acc = mus.evaluate_seed_clustering(
        seeds_cluster, full_df, np.array(unique_terms)
    )

    # Also extract raw predictions for audit reuse
    term_to_cluster = {}
    for cid, (seed, members) in enumerate(seeds_cluster.items()):
        for term in members:
            term_to_cluster[term] = cid

    c1 = np.array([term_to_cluster.get(t, -1) for t in terms1])
    c2 = np.array([term_to_cluster.get(t, -1) for t in terms2])
    preds = ((c1 == c2) & (c1 != -1)).astype(int)

    seeded_rows.append({
        "Model": config.MODELS[model_name]["display_name"],
        "Seeds": params["n_initial_seeds"], "Threshold": f"{params['threshold']:.2f}",
        "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "F1": f"{f1:.4f}",
        "Pos Acc": f"{pos_acc:.4f}", "Neg Acc": f"{neg_acc:.4f}",
    })

    preds_store["seeded"][model_name] = {"preds": preds, "term_to_cluster": term_to_cluster}

display(Markdown("### Results — Seeded Clustering"))
display(pd.DataFrame(seeded_rows).set_index("Model"))


### Results — Seeded Clustering

,Seeds,Threshold,Precision,Recall,F1,Pos Acc,Neg Acc
Model,,,,,,,
All-MPNet-Base-v2,250,0.70,0.3672,0.4503,0.4045,0.4503,0.9997
MPNet-Personality,10,0.70,0.4874,0.3745,0.4236,0.3745,0.9999
SciBERT (SciVocab),250,0.85,0.0138,0.2596,0.0262,0.2596,0.9933
BERT Base,250,0.85,0.0182,0.2013,0.0334,0.2013,0.9961


---

## 6. Audit 1 — Reliability (Stability Test)

**Objective:** Measure the Standard Error of Model (SEM) under stochastic embedding noise.

**Method:**
1. For each noise level $\sigma_\epsilon \in \{0, 0.05, 0.1, 0.2, 0.3\}$, inject Gaussian noise: $\tilde{e} = e + \sigma_\epsilon \cdot \hat\sigma \cdot z$, where $\hat\sigma$ is the per-dimension std and $z \sim \mathcal{N}(0, 1)$.
2. Re-run the technique $k = 5$ times per noise level.
3. Compute $\text{ICC}(1,1)$ across runs and $\text{SEM} = \text{SD}_{F_1} \cdot \sqrt{1 - \text{ICC}}$.

Lower SEM indicates a more stable model.

In [11]:
from sklearn.preprocessing import normalize

def _run_reliability_for_technique(technique, model_name, emb_cache,
                                   noise_levels, n_runs):
    """Run reliability audit for one model × one technique."""
    emb_raw = emb_cache["raw_np"]
    sigma   = emb_cache["sigma"]

    params_c = BEST_CLUSTERING_PARAMS.get(model_name, {})
    thr_pw   = BEST_PAIRWISE_THRESHOLDS.get(model_name, 0.5)
    params_s = BEST_SEEDED_PARAMS.get(model_name, {})

    results = {}
    for nl in noise_levels:
        f1_runs, all_preds = [], []
        for run in range(n_runs):
            rng = np.random.default_rng(config.SEED + run + int(nl * 10000))
            noisy = emb_raw + (rng.normal(0, 1, emb_raw.shape) * (nl * sigma)) if nl > 0 else emb_raw
            noisy_norm_np = normalize(noisy, norm="l2", axis=1)
            noisy_norm_np = np.ascontiguousarray(noisy_norm_np, dtype=np.float64)
            noisy_norm_t  = torch.nn.functional.normalize(
                torch.tensor(noisy, dtype=torch.float32), p=2, dim=1
            )

            if technique == "clustering":
                # ── Use muc.reduce_embeddings_with_umap + muc.run_hdbscan_clustering
                #    + muc.evaluate_clustering (same functions as Section 5.1)
                n_comp = params_c.get("n_components")
                if n_comp is None or n_comp >= noisy_norm_np.shape[1]:
                    red = noisy_norm_np
                else:
                    red, _ = muc.reduce_embeddings_with_umap(
                        noisy_norm_np, n_comp, random_state=config.SEED + run
                    )
                cl = muc.run_hdbscan_clustering(
                    red, params_c["min_cluster_size"], params_c["min_samples"]
                )
                _, _, f1_val, _, _, _, _ = muc.evaluate_clustering(
                    cl, full_df, np.array(unique_terms)
                )
                # Reconstruct preds for ICC computation
                p_ = ((cl[idx1] != -1) & (cl[idx1] == cl[idx2])).astype(int)

            elif technique == "pairwise":
                # ── Use mup.compute_similarities + mup.evaluate_pairwise
                #    (same functions as Section 5.2)
                sims = mup.compute_similarities(noisy_norm_t, term_to_idx, terms1, terms2)
                _, _, f1_val, _, _, _ = mup.evaluate_pairwise(sims, labels, thr_pw)
                p_ = (sims > thr_pw).astype(np.int8)

            else:  # seeded
                # ── Use mus.sample_unique_terms + mus.seed_clustering
                #    + mus.evaluate_seed_clustering (same functions as Section 5.3)
                nte = noisy_norm_t.cpu().detach()
                t2e = {t: e for t, e in zip(unique_terms, nte)}
                seeds, rem = mus.sample_unique_terms(
                    params_s.get("n_initial_seeds", 250), full_df,
                    np.array(unique_terms), random_state=config.SEED + run
                )
                with redirect_stdout(StringIO()):
                    sc = mus.seed_clustering(
                        seeds, rem, t2e, threshold=params_s.get("threshold", 0.7)
                    )
                _, _, f1_val, _, _ = mus.evaluate_seed_clustering(
                    sc, full_df, np.array(unique_terms)
                )
                # Reconstruct preds for ICC computation
                t2c = {}
                for cid, (sd, ms) in enumerate(sc.items()):
                    for m in ms:
                        t2c[m] = cid
                c1 = np.array([t2c.get(t, -1) for t in terms1])
                c2 = np.array([t2c.get(t, -1) for t in terms2])
                p_ = ((c1 == c2) & (c1 != -1)).astype(int)

            f1_runs.append(f1_val)
            all_preds.append(p_)

        f1_arr = np.array(f1_runs)
        preds_mat = np.array(all_preds)
        icc = shared.compute_icc_oneway(
            preds_mat.sum(axis=0), (preds_mat ** 2).sum(axis=0),
            preds_mat.shape[1], n_runs
        )
        sd = np.std(f1_arr, ddof=1)
        sem = sd * np.sqrt(max(0, 1 - icc))
        results[nl] = {"mean_f1": f1_arr.mean(), "sd_f1": sd, "icc": icc, "sem": sem}
    return results


# ── Run for all techniques ───────────────────────────────────────────────────
audit1_records = []
for technique in ["clustering", "pairwise", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        if model_name not in embedding_cache:
            continue
        print(f"  Reliability | {technique:10s} | {model_name} ...", end="", flush=True)
        res = _run_reliability_for_technique(
            technique, model_name, embedding_cache[model_name],
            NOISE_LEVELS, N_RELIABILITY_RUNS
        )
        print(" ✓")
        for nl, vals in res.items():
            audit1_records.append({
                "Technique": technique,
                "Model": config.MODELS[model_name]["display_name"],
                "Noise σ": nl,
                "Mean F1": f"{vals['mean_f1']:.4f}",
                "SD(F1)": f"{vals['sd_f1']:.4f}",
                "ICC": f"{vals['icc']:.4f}",
                "SEM": f"{vals['sem']:.4f}",
            })

df_audit1 = pd.DataFrame(audit1_records)
display(Markdown("### Audit 1 — Reliability Results"))

for tech in ["clustering", "pairwise", "seeded"]:
    display(Markdown(f"#### {tech.upper()}"))
    sub = df_audit1[df_audit1["Technique"] == tech].drop(columns="Technique")

    # Full table: all metrics per model × noise level
    display(sub.set_index(["Model", "Noise σ"]))

    # Compact pivot: Mean F1 per model × noise level
    display(Markdown("*Mean F1 pivot (rows = Model, cols = Noise σ):*"))
    pivot = sub.pivot(index="Model", columns="Noise σ", values="Mean F1")
    display(pivot)


  Reliability | clustering | all-mpnet-base-v2 ... ✓
  Reliability | clustering | dwulff/mpnet-personality ... ✓
  Reliability | clustering | allenai/scibert_scivocab_uncased ... ✓
  Reliability | clustering | bert-base-uncased ... ✓
  Reliability | pairwise   | all-mpnet-base-v2 ... ✓
  Reliability | pairwise   | dwulff/mpnet-personality ... ✓
  Reliability | pairwise   | allenai/scibert_scivocab_uncased ... ✓
  Reliability | pairwise   | bert-base-uncased ... ✓
  Reliability | seeded     | all-mpnet-base-v2 ... ✓
  Reliability | seeded     | dwulff/mpnet-personality ... ✓
  Reliability | seeded     | allenai/scibert_scivocab_uncased ... ✓
  Reliability | seeded     | bert-base-uncased ... ✓


### Audit 1 — Reliability Results

#### CLUSTERING

Mean F1  SD(F1)     ICC     SEM
Model              Noise σ                                
All-MPNet-Base-v2  0.00     0.4758  0.0000  1.0000  0.0000
                   0.05     0.4757  0.0013  0.9821  0.0002
                   0.10     0.4763  0.0011  0.9641  0.0002
                   0.20     0.4770  0.0030  0.9229  0.0008
                   0.30     0.4785  0.0038  0.8803  0.0013
MPNet-Personality  0.00     0.4798  0.0000  1.0000  0.0000
                   0.05     0.4773  0.0016  0.9801  0.0002
                   0.10     0.4765  0.0013  0.9625  0.0003
                   0.20     0.4742  0.0022  0.9272  0.0006
                   0.30     0.4718  0.0022  0.8917  0.0007
SciBERT (SciVocab) 0.00     0.2382  0.0000  1.0000  0.0000
                   0.05     0.2391  0.0018  0.9570  0.0004
                   0.10     0.2381  0.0038  0.9180  0.0011
                   0.20     0.2351  0.0033  0.8604  0.0012
                   0.30     0.2304  0.0037  0.8106  0.0016
BERT Base          0.00     0.2241  0.0000  1.0000  0.0000
                   0.05     0.2224  0.0012  0.9583  0.0003
                   0.10     0.2225  0.0013  0.9229  0.0004
                   0.20     0.2149  0.0076  0.8177  0.0032
                   0.30     0.2063  0.0082  0.7484  0.0041

*Mean F1 pivot (rows = Model, cols = Noise σ):*

...

Noise σ,0.00,0.05,0.10,0.20,0.30
Model,,,,,
All-MPNet-Base-v2,0.4654,0.4664,0.4705,0.4836,0.4852
BERT Base,0.0871,0.0883,0.0911,0.0964,0.0749
MPNet-Personality,0.4706,0.4709,0.4726,0.4764,0.4638
SciBERT (SciVocab),0.1621,0.1635,0.1696,0.1875,0.1918


#### SEEDED

Mean F1  SD(F1)     ICC     SEM
Model              Noise σ                                
All-MPNet-Base-v2  0.00     0.3979  0.0070  0.8559  0.0027
                   0.05     0.3999  0.0071  0.8485  0.0028
                   0.10     0.4023  0.0050  0.8388  0.0020
                   0.20     0.4192  0.0074  0.8245  0.0031
                   0.30     0.4284  0.0059  0.8211  0.0025
MPNet-Personality  0.00     0.4238  0.0001  0.9985  0.0000
                   0.05     0.4233  0.0010  0.9812  0.0001
                   0.10     0.4228  0.0016  0.9673  0.0003
                   0.20     0.4153  0.0022  0.9294  0.0006
                   0.30     0.3945  0.0025  0.9129  0.0008
SciBERT (SciVocab) 0.00     0.0286  0.0016  0.4231  0.0012
                   0.05     0.0283  0.0016  0.4284  0.0012
                   0.10     0.0293  0.0015  0.4126  0.0011
                   0.20     0.0275  0.0022  0.4419  0.0017
                   0.30     0.0274  0.0016  0.4761  0.0012
BERT Base          0.00     0.0407  0.0046  0.4806  0.0033
                   0.05     0.0403  0.0048  0.4866  0.0034
                   0.10     0.0401  0.0033  0.4846  0.0024
                   0.20     0.0375  0.0037  0.5514  0.0024
                   0.30     0.0332  0.0045  0.5695  0.0030

*Mean F1 pivot (rows = Model, cols = Noise σ):*

Noise σ,0.00,0.05,0.10,0.20,0.30
Model,,,,,
All-MPNet-Base-v2,0.3979,0.3999,0.4023,0.4192,0.4284
BERT Base,0.0407,0.0403,0.0401,0.0375,0.0332
MPNet-Personality,0.4238,0.4233,0.4228,0.4153,0.3945
SciBERT (SciVocab),0.0286,0.0283,0.0293,0.0275,0.0274


---

## 7. Audit 2 — Discriminant Validity (Lexical Trap)

**Objective:** Detect whether models are fooled by surface-level orthographic similarity.

We evaluate models on two types of negative pairs (label = 0 only):

1. **Hard Negatives (Lexical Traps)**: Negative pairs whose character 3-gram Jaccard similarity exceeds 0.5 (high surface similarity but semantically unrelated).
2. **Easy Negatives**: Negative pairs with zero lexical overlap (Jaccard = 0.0).

Because both subsets contain **only label=0 pairs**, Precision / Recall / F1 (which treat label=1 as positive) are undefined and always 0. The sole meaningful metric is:

- **FPR** (False Positive Rate): $\text{FPR} = \frac{FP}{FP + TN}$

Lower FPR on hard negatives means the model correctly rejects unrelated terms despite high letter overlap (good discriminant validity).


In [12]:
audit2_records = []

for technique in ["clustering", "pairwise", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        info = preds_store[technique].get(model_name)
        if info is None:
            continue
        preds = info["preds"]

        # Evaluate on hard negatives (lexical traps)
        hard_metrics = shared.compute_subset_metrics(labels, preds, hard_neg_mask)
        audit2_records.append({
            "Technique": technique,
            "Model": config.MODELS[model_name]["display_name"],
            "Subset": "Hard Neg (Jacc>0.5)",
            "N": hard_metrics['n_subset'],
            "FPR": f"{hard_metrics['fpr']:.4f}",
            "TN": hard_metrics['tn'],
            "FP": hard_metrics['fp'],
        })

        # Evaluate on easy negatives (zero overlap)
        easy_metrics = shared.compute_subset_metrics(labels, preds, easy_neg_mask)
        audit2_records.append({
            "Technique": technique,
            "Model": config.MODELS[model_name]["display_name"],
            "Subset": "Easy Neg (Jacc=0.0)",
            "N": easy_metrics['n_subset'],
            "FPR": f"{easy_metrics['fpr']:.4f}",
            "TN": easy_metrics['tn'],
            "FP": easy_metrics['fp'],
        })

display(Markdown("### Audit 2 — Discriminant Validity Results"))
display(Markdown("""
**Hard Negatives (Lexical Traps)** are negative pairs with high surface similarity (Jaccard > 0.5).  
**Easy Negatives** are negative pairs with zero lexical overlap (Jaccard = 0.0).  

> **Note:** These subsets contain only label=0 pairs (true negatives), so Precision / Recall / F1  
> (which treat label=1 as positive) are undefined and always 0. Only **FPR** is the meaningful metric here:  
> lower FPR means the model correctly rejects more lexically-similar but unrelated pairs.
"""))
display(pd.DataFrame(audit2_records).set_index(["Technique", "Model", "Subset"]))


### Audit 2 — Discriminant Validity Results


**Hard Negatives (Lexical Traps)** are negative pairs with high surface similarity (Jaccard > 0.5).  
**Easy Negatives** are negative pairs with zero lexical overlap (Jaccard = 0.0).  

> **Note:** These subsets contain only label=0 pairs (true negatives), so Precision / Recall / F1  
> (which treat label=1 as positive) are undefined and always 0. Only **FPR** is the meaningful metric here:  
> lower FPR means the model correctly rejects more lexically-similar but unrelated pairs.


N     FPR        TN  \
Technique  Model              Subset                                            
clustering All-MPNet-Base-v2  Hard Neg (Jacc>0.5)       932  0.1931       752   
                              Easy Neg (Jacc=0.0)  12886583  0.0000  12886288   
           MPNet-Personality  Hard Neg (Jacc>0.5)       932  0.1856       759   
                              Easy Neg (Jacc=0.0)  12886583  0.0000  12886313   
           SciBERT (SciVocab) Hard Neg (Jacc>0.5)       932  0.2200       727   
                              Easy Neg (Jacc=0.0)  12886583  0.0000  12886244   
           BERT Base          Hard Neg (Jacc>0.5)       932  0.2006       745   
                              Easy Neg (Jacc=0.0)  12886583  0.0000  12886294   
pairwise   All-MPNet-Base-v2  Hard Neg (Jacc>0.5)       932  0.3262       628   
                              Easy Neg (Jacc=0.0)  12886583  0.0000  12886047   
           MPNet-Personality  Hard Neg (Jacc>0.5)       932  0.2929       659   
                              Easy Neg (Jacc=0.0)  12886583  0.0000  12886388   
           SciBERT (SciVocab) Hard Neg (Jacc>0.5)       932  0.3155       638   
                              Easy Neg (Jacc=0.0)  12886583  0.0003  12882628   
           BERT Base          Hard Neg (Jacc>0.5)       932  0.1770       767   
                              Easy Neg (Jacc=0.0)  12886583  0.0006  12879401   
seeded     All-MPNet-Base-v2  Hard Neg (Jacc>0.5)       932  0.2414       707   
                              Easy Neg (Jacc=0.0)  12886583  0.0001  12885296   
           MPNet-Personality  Hard Neg (Jacc>0.5)       932  0.2114       735   
                              Easy Neg (Jacc=0.0)  12886583  0.0000  12886358   
           SciBERT (SciVocab) Hard Neg (Jacc>0.5)       932  0.3680       589   
                              Easy Neg (Jacc=0.0)  12886583  0.0059  12810549   
           BERT Base          Hard Neg (Jacc>0.5)       932  0.3015       651   
                              Easy Neg (Jacc=0.0)  12886583  0.0037  12838846   

                                                      FP  
Technique  Model              Subset                      
clustering All-MPNet-Base-v2  Hard Neg (Jacc>0.5)    180  
                              Easy Neg (Jacc=0.0)    295  
           MPNet-Personality  Hard Neg (Jacc>0.5)    173  
                              Easy Neg (Jacc=0.0)    270  
           SciBERT (SciVocab) Hard Neg (Jacc>0.5)    205  
                              Easy Neg (Jacc=0.0)    339  
           BERT Base          Hard Neg (Jacc>0.5)    187  
                              Easy Neg (Jacc=0.0)    289  
pairwise   All-MPNet-Base-v2  Hard Neg (Jacc>0.5)    304  
                              Easy Neg (Jacc=0.0)    536  
           MPNet-Personality  Hard Neg (Jacc>0.5)    273  
                              Easy Neg (Jacc=0.0)    195  
           SciBERT (SciVocab) Hard Neg (Jacc>0.5)    294  
                              Easy Neg (Jacc=0.0)   3955  
           BERT Base          Hard Neg (Jacc>0.5)    165  
                              Easy Neg (Jacc=0.0)   7182  
seeded     All-MPNet-Base-v2  Hard Neg (Jacc>0.5)    225  
                              Easy Neg (Jacc=0.0)   1287  
           MPNet-Personality  Hard Neg (Jacc>0.5)    197  
                              Easy Neg (Jacc=0.0)    225  
           SciBERT (SciVocab) Hard Neg (Jacc>0.5)    343  
                              Easy Neg (Jacc=0.0)  76034  
           BERT Base          Hard Neg (Jacc>0.5)    281  
                              Easy Neg (Jacc=0.0)  47737

---

## 8. Audit 3 — Differential Item Functioning (Rare-Word Bias)

**Objective:** Detect systematic bias against rare / technical terminology.

$$\Delta\text{Recall} = \text{Recall}_{\text{Common}} - \text{Recall}_{\text{Rare}}$$

- **Common pairs**: top 10 % of Zipf frequency.  
- **Rare pairs**: bottom 10 % of Zipf frequency.  
- A large positive $\Delta$Recall indicates the model underperforms on rare terms.

In [13]:
if not has_wordfreq:
    display(Markdown("**⚠ Skipped** — `wordfreq` library is not installed."))
else:
    audit3_records = []
    for technique in ["clustering", "pairwise", "seeded"]:
        for model_name in MODELS_TO_EVAL:
            preds = preds_store[technique].get(model_name, {}).get("preds")
            if preds is None:
                continue
            rec_common = shared.recall_on_subset(labels, preds, common_mask)
            rec_rare   = shared.recall_on_subset(labels, preds, rare_mask)
            gap = rec_common - rec_rare
            audit3_records.append({
                "Technique": technique,
                "Model": config.MODELS[model_name]["display_name"],
                "Recall (Common)": f"{rec_common:.4f}",
                "Recall (Rare)": f"{rec_rare:.4f}",
                "ΔRecall": f"{gap:+.4f}",
                "N Common": int(common_mask.sum()),
                "N Rare": int(rare_mask.sum()),
            })

    display(Markdown("### Audit 3 — DIF Results"))
    display(pd.DataFrame(audit3_records).set_index(["Technique", "Model"]))


### Audit 3 — DIF Results

Recall (Common) Recall (Rare)  ΔRecall  \
Technique  Model                                                       
clustering All-MPNet-Base-v2           0.5327        0.2257  +0.3070   
           MPNet-Personality           0.5310        0.2187  +0.3123   
           SciBERT (SciVocab)          0.2655        0.1429  +0.1226   
           BERT Base                   0.2212        0.1693  +0.0519   
pairwise   All-MPNet-Base-v2           0.6602        0.2557  +0.4044   
           MPNet-Personality           0.6035        0.2257  +0.3778   
           SciBERT (SciVocab)          0.3062        0.1587  +0.1475   
           BERT Base                   0.1770        0.0899  +0.0870   
seeded     All-MPNet-Base-v2           0.6071        0.2257  +0.3813   
           MPNet-Personality           0.4938        0.1711  +0.3227   
           SciBERT (SciVocab)          0.3168        0.2416  +0.0752   
           BERT Base                   0.2549        0.1517  +0.1032   

                               N Common  N Rare  
Technique  Model                                 
clustering All-MPNet-Base-v2        565     567  
           MPNet-Personality        565     567  
           SciBERT (SciVocab)       565     567  
           BERT Base                565     567  
pairwise   All-MPNet-Base-v2        565     567  
           MPNet-Personality        565     567  
           SciBERT (SciVocab)       565     567  
           BERT Base                565     567  
seeded     All-MPNet-Base-v2        565     567  
           MPNet-Personality        565     567  
           SciBERT (SciVocab)       565     567  
           BERT Base                565     567

---

## 9. Audit 4 — Semantic Decay (Semantic Gap Test)

**Objective:** Test robustness as keyword overlap vanishes.

$$\text{Slope} = \text{Recall}_{\text{Hard}} - \text{Recall}_{\text{Easy}}$$

- **Easy positives**: Jaccard > 0.5 (lexical anchors — the model can "cheat" with surface cues).  
- **Hard positives**: Jaccard = 0.0 (semantic gap — requires genuine semantic understanding).  
- A negative slope indicates performance degrades on purely-semantic pairs.

In [14]:
audit4_records = []

for technique in ["clustering", "pairwise", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        preds = preds_store[technique].get(model_name, {}).get("preds")
        if preds is None:
            continue
        rec_easy = shared.recall_on_subset(labels, preds, easy_mask)
        rec_hard = shared.recall_on_subset(labels, preds, hard_mask)
        slope = rec_hard - rec_easy
        audit4_records.append({
            "Technique": technique,
            "Model": config.MODELS[model_name]["display_name"],
            "Recall (Easy)": f"{rec_easy:.4f}",
            "Recall (Hard)": f"{rec_hard:.4f}",
            "Slope": f"{slope:+.4f}",
            "N Easy": int(easy_mask.sum()),
            "N Hard": int(hard_mask.sum()),
        })

display(Markdown("### Audit 4 — Semantic Decay Results"))
display(pd.DataFrame(audit4_records).set_index(["Technique", "Model"]))


### Audit 4 — Semantic Decay Results

Recall (Easy) Recall (Hard)    Slope  N Easy  \
Technique  Model                                                             
clustering All-MPNet-Base-v2         0.8444        0.0968  -0.7476    1086   
           MPNet-Personality         0.8600        0.1037  -0.7564    1086   
           SciBERT (SciVocab)        0.3250        0.0256  -0.2995    1086   
           BERT Base                 0.3766        0.0249  -0.3517    1086   
pairwise   All-MPNet-Base-v2         0.9273        0.1493  -0.7780    1086   
           MPNet-Personality         0.8978        0.1030  -0.7948    1086   
           SciBERT (SciVocab)        0.3508        0.0290  -0.3218    1086   
           BERT Base                 0.2882        0.0242  -0.2640    1086   
seeded     All-MPNet-Base-v2         0.8517        0.1272  -0.7246    1086   
           MPNet-Personality         0.8112        0.0753  -0.7359    1086   
           SciBERT (SciVocab)        0.3204        0.0878  -0.2327    1086   
           BERT Base                 0.3941        0.0463  -0.3478    1086   

                               N Hard  
Technique  Model                       
clustering All-MPNet-Base-v2     1447  
           MPNet-Personality     1447  
           SciBERT (SciVocab)    1447  
           BERT Base             1447  
pairwise   All-MPNet-Base-v2     1447  
           MPNet-Personality     1447  
           SciBERT (SciVocab)    1447  
           BERT Base             1447  
seeded     All-MPNet-Base-v2     1447  
           MPNet-Personality     1447  
           SciBERT (SciVocab)    1447  
           BERT Base             1447

---

## 10. Audit 5 — Structural Validity (Map Match)

**Objective:** Assess whether model-derived distances correlate with expert-curated APA graph distances.
We compute three correlation coefficients:

| Correlation | Description |
|------------|-------------|
| **Spearman ρ** | Rank-order correlation (robust to non-linear monotonic relationships) |
| **Pearson r** | Linear correlation between distances |
| **Point-biserial r** | Correlation between a binary model variable (same cluster = 0, diff = 1) and continuous expert distance |

For **clustering / seeded**: binary distance (0 = same cluster, 1 = different).
For **pairwise**: binary component membership, model-graph shortest path, and cosine distance.

In [15]:
expert_mask = shortest_path >= 0
d_expert = shortest_path[expert_mask].astype(float)
print(f"Pairs with valid expert distances: {expert_mask.sum():,}\n")

audit5_records = []

# ── Clustering & Seeded: binary model distance ──────────────────────────────
for technique in ["clustering", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        info = preds_store[technique].get(model_name)
        if info is None:
            continue

        if technique == "clustering":
            cl = info["cluster_labels"]
            l1, l2 = cl[info["idx1"]], cl[info["idx2"]]
            d_model = np.where((l1 != -1) & (l1 == l2), 0, 1).astype(float)
        else:
            t2c = info["term_to_cluster"]
            c1 = np.array([t2c.get(t, -1) for t in terms1])
            c2 = np.array([t2c.get(t, -1) for t in terms2])
            d_model = np.where((c1 != -1) & (c1 == c2), 0, 1).astype(float)

        d_m = d_model[expert_mask]
        rho_s = shared.spearman_corr(d_expert, d_m)
        rho_p, p_val_p = shared.pearson_corr(d_expert, d_m)
        rpb, p_val_pb = shared.point_biserial_corr(d_m, d_expert)

        audit5_records.append({
            "Technique": technique,
            "Model": config.MODELS[model_name]["display_name"],
            "Metric": "binary_dist",
            "Spearman ρ": f"{rho_s:.4f}",
            "Pearson r": f"{rho_p:.4f}",
            "Point-Biserial r": f"{rpb:.4f}" if not np.isnan(rpb) else "N/A",
            "p (Pearson)": f"{p_val_p:.2e}" if not np.isnan(p_val_p) else "N/A",
            "p (PB)": f"{p_val_pb:.2e}" if not np.isnan(p_val_pb) else "N/A",
        })

# ── Pairwise: multiple distance metrics ─────────────────────────────────────
for model_name in MODELS_TO_EVAL:
    info = preds_store["pairwise"].get(model_name)
    if info is None:
        continue
    preds = info["preds"]
    sims  = info["similarities"]

    # (a) Binary same-component via union-find
    edges_mask = preds == 1
    parent, find, union_fn = shared.union_find(len(unique_terms))
    for a, b in zip(idx1[edges_mask], idx2[edges_mask]):
        union_fn(a, b)
    roots1 = np.array([find(i) for i in idx1])
    roots2 = np.array([find(i) for i in idx2])
    same_comp = (roots1 == roots2).astype(float)

    rho_s  = shared.spearman_corr(d_expert, same_comp[expert_mask])
    rho_p, pp = shared.pearson_corr(d_expert, same_comp[expert_mask])
    rpb, ppb  = shared.point_biserial_corr(same_comp[expert_mask], d_expert)

    audit5_records.append({
        "Technique": "pairwise", "Model": config.MODELS[model_name]["display_name"],
        "Metric": "binary_component",
        "Spearman ρ": f"{rho_s:.4f}", "Pearson r": f"{rho_p:.4f}",
        "Point-Biserial r": f"{rpb:.4f}" if not np.isnan(rpb) else "N/A",
        "p (Pearson)": f"{pp:.2e}" if not np.isnan(pp) else "N/A",
        "p (PB)": f"{ppb:.2e}" if not np.isnan(ppb) else "N/A",
    })

    # (b) Model-graph shortest path
    model_sp = shared.model_graph_shortest_paths(
        len(unique_terms), idx1[edges_mask], idx2[edges_mask], idx1, idx2)
    valid_sp = expert_mask & (model_sp >= 0)
    rho_sp = shared.spearman_corr(shortest_path[valid_sp].astype(float), model_sp[valid_sp].astype(float))
    rp_sp, pp_sp = shared.pearson_corr(shortest_path[valid_sp].astype(float), model_sp[valid_sp].astype(float))

    audit5_records.append({
        "Technique": "pairwise", "Model": config.MODELS[model_name]["display_name"],
        "Metric": "graph_shortest_path",
        "Spearman ρ": f"{rho_sp:.4f}", "Pearson r": f"{rp_sp:.4f}",
        "Point-Biserial r": "N/A (continuous)",
        "p (Pearson)": f"{pp_sp:.2e}" if not np.isnan(pp_sp) else "N/A",
        "p (PB)": "—",
    })

    # (c) Cosine distance
    cos_dist = 1.0 - sims
    rho_cd = shared.spearman_corr(d_expert, cos_dist[expert_mask])
    rp_cd, pp_cd = shared.pearson_corr(d_expert, cos_dist[expert_mask])

    audit5_records.append({
        "Technique": "pairwise", "Model": config.MODELS[model_name]["display_name"],
        "Metric": "cosine_distance",
        "Spearman ρ": f"{rho_cd:.4f}", "Pearson r": f"{rp_cd:.4f}",
        "Point-Biserial r": "N/A (continuous)",
        "p (Pearson)": f"{pp_cd:.2e}" if not np.isnan(pp_cd) else "N/A",
        "p (PB)": "—",
    })

display(Markdown("### Audit 5 — Structural Validity Results"))
display(pd.DataFrame(audit5_records).set_index(["Technique", "Model", "Metric"]))


Pairs with valid expert distances: 15,387,276



### Audit 5 — Structural Validity Results

Spearman ρ Pearson r  \
Technique  Model              Metric                                     
clustering All-MPNet-Base-v2  binary_dist             0.0235    0.0431   
           MPNet-Personality  binary_dist             0.0223    0.0415   
           SciBERT (SciVocab) binary_dist             0.0215    0.0351   
           BERT Base          binary_dist             0.0200    0.0327   
seeded     All-MPNet-Base-v2  binary_dist             0.0297    0.0489   
           MPNet-Personality  binary_dist             0.0220    0.0399   
           SciBERT (SciVocab) binary_dist             0.0566    0.0659   
           BERT Base          binary_dist             0.0319    0.0378   
pairwise   All-MPNet-Base-v2  binary_component       -0.0115   -0.0139   
                              graph_shortest_path     0.2384    0.2714   
                              cosine_distance         0.2629    0.2902   
           MPNet-Personality  binary_component       -0.0182   -0.0253   
                              graph_shortest_path     0.3752    0.4215   
                              cosine_distance         0.1712    0.2110   
           SciBERT (SciVocab) binary_component       -0.0133   -0.0156   
                              graph_shortest_path     0.1156    0.1304   
                              cosine_distance         0.1868    0.1927   
           BERT Base          binary_component       -0.0308   -0.0333   
                              graph_shortest_path     0.1241    0.0626   
                              cosine_distance         0.2902    0.2967   

                                                   Point-Biserial r  \
Technique  Model              Metric                                  
clustering All-MPNet-Base-v2  binary_dist                    0.0431   
           MPNet-Personality  binary_dist                    0.0415   
           SciBERT (SciVocab) binary_dist                    0.0351   
           BERT Base          binary_dist                    0.0327   
seeded     All-MPNet-Base-v2  binary_dist                    0.0489   
           MPNet-Personality  binary_dist                    0.0399   
           SciBERT (SciVocab) binary_dist                    0.0659   
           BERT Base          binary_dist                    0.0378   
pairwise   All-MPNet-Base-v2  binary_component              -0.0139   
                              graph_shortest_path  N/A (continuous)   
                              cosine_distance      N/A (continuous)   
           MPNet-Personality  binary_component              -0.0253   
                              graph_shortest_path  N/A (continuous)   
                              cosine_distance      N/A (continuous)   
           SciBERT (SciVocab) binary_component              -0.0156   
                              graph_shortest_path  N/A (continuous)   
                              cosine_distance      N/A (continuous)   
           BERT Base          binary_component              -0.0333   
                              graph_shortest_path  N/A (continuous)   
                              cosine_distance      N/A (continuous)   

                                                  p (Pearson)    p (PB)  
Technique  Model              Metric                                     
clustering All-MPNet-Base-v2  binary_dist            0.00e+00  0.00e+00  
           MPNet-Personality  binary_dist            0.00e+00  0.00e+00  
           SciBERT (SciVocab) binary_dist            0.00e+00  0.00e+00  
           BERT Base          binary_dist            0.00e+00  0.00e+00  
seeded     All-MPNet-Base-v2  binary_dist            0.00e+00  0.00e+00  
           MPNet-Personality  binary_dist            0.00e+00  0.00e+00  
           SciBERT (SciVocab) binary_dist            0.00e+00  0.00e+00  
           BERT Base          binary_dist            0.00e+00  0.00e+00  
pairwise   All-MPNet-Base-v2  binary_component       0.00e+00  0.00e+00  
                              graph_shortest

---

## 11. Consolidated Summary

A single overview table combining the core F1 metric across all models and techniques, plus a per-technique best-model highlight.

In [16]:
# ── Build consolidated F1 summary ─────────────────────────────────────────────
summary_rows = []

for tech_name, rows_list in [("Clustering", clustering_rows),
                              ("Pairwise", pairwise_rows),
                              ("Seeded", seeded_rows)]:
    for row in rows_list:
        summary_rows.append({
            "Technique": tech_name,
            "Model": row["Model"],
            "Precision": row["Precision"],
            "Recall": row["Recall"],
            "F1": row["F1"],
        })

df_summary = pd.DataFrame(summary_rows)

display(Markdown("### F1 Summary — All Models × All Techniques"))
pivot_f1 = df_summary.pivot(index="Model", columns="Technique", values="F1")
display(pivot_f1)

# ── Best model per technique ─────────────────────────────────────────────────
display(Markdown("### Best Model per Technique"))
for tech in ["Clustering", "Pairwise", "Seeded"]:
    sub = df_summary[df_summary["Technique"] == tech]
    best = sub.loc[sub["F1"].astype(float).idxmax()]
    print(f"  {tech:12s} → {best['Model']}  (F1 = {best['F1']})")

# ── Save summary to CSV ─────────────────────────────────────────────────────
import os
os.makedirs("results/apa", exist_ok=True)
summary_path = "results/apa/final_test_summary.csv"
df_summary.to_csv(summary_path, index=False)
print(f"\n✓ Summary saved to {summary_path}")


### F1 Summary — All Models × All Techniques

Technique,Clustering,Pairwise,Seeded
Model,,,
All-MPNet-Base-v2,0.4758,0.4654,0.4045
BERT Base,0.2241,0.0871,0.0334
MPNet-Personality,0.4798,0.4706,0.4236
SciBERT (SciVocab),0.2382,0.1621,0.0262


### Best Model per Technique

  Clustering   → MPNet-Personality  (F1 = 0.4798)
  Pairwise     → MPNet-Personality  (F1 = 0.4706)
  Seeded       → MPNet-Personality  (F1 = 0.4236)

✓ Summary saved to results/apa/final_test_summary.csv


---

## Reproducibility Notes

| Item | Value |
|------|-------|
| Random seed | `SEED = 42` across all operations |
| Test data | `datasets/processed_datasets/test_{positive,negative}_pairs.csv` |
| Models | Loaded from local HuggingFace cache (`local_files_only=True`) |
| Hyperparameters | Fixed from prior cross-validation (Section 1) |
| Correlations | Spearman ρ, Pearson r, and Point-Biserial r (Audit 3) |

**To reproduce:** Run all cells top-to-bottom in a fresh kernel.  
All utility functions live in `model_utils_shared.py`, `model_utils_clustering.py`, `model_utils_pairwise.py`, and `model_utils_seed.py`.